# Rennaigan backend on Colab

Runs the five detector services and the gateway on a Colab GPU and gives you a public
`https://….trycloudflare.com` address to paste into the UI (**Settings → ML Gateway → Gateway URL**).

Before running: **Runtime → Change runtime type → T4 GPU**. Then **Runtime → Run all**.
The first run takes about 8 minutes (model downloads). Keep this tab open while you use the UI.

If a cell fails, copy its full output; the self-test cell near the end reports each detector separately.

After a code update: re-run cells 3 and 4. Cell 3 pulls the latest code and restarts the backend; the public address from cell 5 stays the same.

In [ ]:
#@title 1. Get the code
REPO = "https://github.com/Laxitha/Rennaigan.git"
BRANCH = "backend-gateway"   # change to "main" once the pull request is merged

import os, subprocess, torch
os.chdir("/content")
if not os.path.exists("/content/Rennaigan"):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/Rennaigan
os.chdir("/content/Rennaigan")
!git pull -q
!git log --oneline -1
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a T4 GPU")

In [ ]:
#@title 2. One-time setup: tools, model repositories, packages and weights (about 6 minutes)
!bash scripts/colab_setup.sh

In [ ]:
#@title 3. Start the detector services and the gateway
import os
os.chdir("/content/Rennaigan")

# Optional: a reasoned real/deepfake verdict written by Claude from the detector findings.
# Add your key under the key icon in Colab's left sidebar as a secret named ANTHROPIC_API_KEY
# (it starts with "sk-ant-") and switch on "Notebook access". Never paste the key into a cell.
try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key.strip()
        # Also send the image (or four video frames) to Claude for a visual review.
        # This shares uploaded media with Anthropic; delete the next line to send detector findings only.
        os.environ["RENNAIGAN_ASSESS_SEND_MEDIA"] = "1"
except Exception:
    pass

!git pull -q
!python scripts/colab.py restart

In [ ]:
#@title 4. Self-test: every detector on a real talking-head clip
!python scripts/colab.py selftest

In [ ]:
#@title 5. Public address for the UI
import re
if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
!pkill -f cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8010", "--no-autoupdate"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1.5)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        url = found.group(0); break
print("Gateway URL:", url or "not ready - run:  !cat /content/tunnel.log")
print("\nPaste it into the UI: Settings -> ML Gateway -> Gateway URL -> Save & Connect.")
print("Anyone with this address can upload to and delete from this gateway. It stops when this Colab session ends.")

In [ ]:
#@title Status and logs (run when something looks wrong)
!python scripts/colab.py status
!tail -15 /content/backend.log